# Day 1-05：ReLUネットワークを学習する
**目標**：`04-relu-functions.ipynb`で手動で調整していたパラメータを，学習で調整すること（機械学習！）を学ぶ



## 学習に必要なもの

学習には、次の3つを順に用意します。

1. **データ**：入力`x`と目標値`y`
2. **モデル**：入力から予測値を計算する関数
3. **最適化**：損失関数、optimizer、訓練用の関数など

## 実行前の準備
> **⚠️ 初回に必ず確認：セル冒頭の`DRIVE_PROJECT`と`LOCAL_PROJECT`を、自分の環境の絶対パスに変更してください。**

このセルを最初に実行してください。Driveのマウント、教材フォルダへの移動、Colab用パッケージの準備を自動で行います。
ローカルではカーネルに **Python (jkj1a)** を選びます。


In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parents[1]
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)


## 1. データの準備

### データを生成する関数の定義

In [ ]:
import io
import torch
from torch import nn
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# 三角形の関数
def triangle(center, width=0.8):
    return torch.clamp(1 - torch.abs(x - center) / width, min=0)

# 台形の関数
def trapezoid(center, half_top=0.25, half_base=0.5):
    y = torch.minimum((x - (center - half_base)) / (half_base - half_top), torch.ones_like(x))
    y = torch.minimum(y, ((center + half_base) - x) / (half_base - half_top))
    return torch.clamp(y, min=0, max=1)

# 目標関数の選択
def target_function(x, target, sin_freq=1):
    if target == "linear": return x
    if target == "triangle": return triangle(0)
    if target == "two_triangles": return triangle(0) + triangle(1)
    if target == "trapezoid": return trapezoid(0)
    if target == "two_trapezoids": return trapezoid(0) + trapezoid(1)
    if target == "sin": return torch.sin(x * sin_freq)
    raise ValueError(target)

### 実際に生成してみる

In [ ]:
x = torch.linspace(-1.5, 2.5, 600)  # 入力 (-1.5 から 2.5 までの600点)
y_target = target_function(x, target='two_triangles') # xに対応する出力を生成

plt.plot(x, y_target); plt.show()

## 2. モデル

1次元入力に対して、次の関数を1層のネットワークとして実装します。

$$f(x)=\sum_{i=1}^{K} w_i\,\operatorname{ReLU}(x-a_i)$$

### モデルの定義．

PyTorchライブラリの`nn.Module`クラスを継承する（雛形にする）ことで，自分で実装する部分は最小限で済む）

**（演習）以下を埋めて，上記の関数を実現するクラスを実装せよ．**

In [ ]:
class ReLUWeightedSum(nn.Module):
    def __init__(self, n_hiddens=4):
        '''
        初期化の関数．最初に一度だけ呼ばれる
        '''
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_hiddens) * (1 / n_hiddens ** 0.5))  # nn.Parameterをつけることで，学習可能なパラメータになる
        self.shift = nn.Parameter(torch.zeros(n_hiddens))

    def forward(self, x):
        '''
        関数に代入した時の振る舞いを定義する．ようはf(x)の計算をここに実装する
        '''
        ... # 上記の関数をここに実装してください（上に書かれている数式をそのまま実装すればOK）


<details><summary>解答例（自分で作ってみてから確認すること）</summary>

```python
class ReLUWeightedSum(nn.Module):
    def __init__(self, n_hiddens=4):
        '''
        初期化の関数．最初に一度だけ呼ばれる
        '''
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_hiddens) * (1 / n_hiddens ** 0.5))  # nn.Parameterをつけることで，学習可能なパラメータになる
        self.shift = nn.Parameter(torch.zeros(n_hiddens))

    def forward(self, x):
        return torch.sum(self.weight * torch.relu(x - self.shift))
```

</details>


### モデルを作る

クラスさえ定義すればこれだけでOK

In [ ]:
model = ReLUWeightedSum(n_hiddens=4)  # 4つのReLUを用意

## 3. 最適化

いわゆる学習を行う．必要なものは以下．
- データ（教材） -- 済✔️
- モデル（生徒） -- 済✔️
- 損失関数（生徒の評価指標）
- optimizer（勾配を用いてパラメータを更新する手法）

以下をよく読むこと！

In [ ]:
## データの準備
x = torch.linspace(-1.5, 2.5, 600)  # 入力 (-1.5 から 2.5 までの600点)
y = target_function(x, target='triangle') # xに対応する出力を生成

data = torch.stack([x, y_target], dim=1)  # xとyを結合．data[0]で一つ目の(x, y)サンプルを取得できる

## モデルを準備
n_hiddens = 4
model = ReLUWeightedSum(n_hiddens=n_hiddens)

## 損失関数の定義（今回は単純な二乗誤差）
def loss_fn(prediction, target):
    return ((prediction - target) ** 2).mean()

## optimizerを準備
learning_rate = 0.05
momentum = 0.9
optimizer = torch.optim.SGD(model.parameters(),  # モデルのパラメータを登録（更新されていく）
                            lr=learning_rate,    # 学習率
                            momentum=momentum)   # 慣性

### `Trainer`クラス

Trainerクラスは上記の4つ（データ・モデル・損失関数・optimizer）を受け取って学習を回すためのクラス．定義したら，以下のように使う．

In [ ]:
## 未定義なので，今実行してもエラーが出る！

trainer = Trainer(model, data, loss_fn, optimizer)  # Trainerを準備（必要なものを渡す）
trainer.train(n_epochs=10)                          # データを10周分訓練
trainer.test(test_data)                                  # 訓練したモデルの最終評価など

以下のように定義する．よく読み込んでほしい（思ったより単純なはず）．特に`train_step`が根幹．

In [ ]:
class Trainer:
    def __init__(self, model, data, loss_fn, optimizer):
        # 初期化．
        self.model = model
        self.data = data
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train_step(self, model, data, loss_fn, optimizer):
        loss_total = 0
        
        for x, y in data:
            optimizer.zero_grad()          # 勾配を0にする
            prediction = model(x)          # モデルの予測f(x)を計算
            loss = loss_fn(prediction, y)  # 予測と目標の誤差を計算 l(f(x), y)
            loss.backward()                # 誤差の勾配を計算 ∂l/∂f(x)
            optimizer.step()               # パラメータを更新 w_i = w_i - η ∂l/∂f(x)∂f(x)/∂w_i, 
                                           #               a_i = a_i - η ∂l/∂f(x)∂f(x)/∂a_i 
                                           
            loss_total += loss.item()

        return loss_total / len(data)      # 平均損失を返す

    def train(self, n_epochs=10, print_interval=100):
        for epoch in range(n_epochs):
            loss = self.train_step(self.model, self.data, self.loss_fn, self.optimizer)
            
            if epoch % print_interval == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def test(self, test_data):
        with torch.no_grad():
            
            loss_total = 0
            for x, y in test_data:
                prediction = self.model(x)
                loss = self.loss_fn(prediction, y)
                loss_total += loss.item()

        return loss_total / len(test_data)

## 5. 学習を実行する

（演習）以下を埋めて，学習を行え．

ここまでで必要なコードは全て登場している．各項目（データの準備，モデルを準備...）ごとに，
1. 思い出しながら，自分の手で雰囲気で埋める
2. 実行するとエラーが出るので上の方にあるコードを探してチェック
という順番で進めること．今回はなるべく自分で手打ちすること！

In [ ]:
## データの準備
... 

## モデルを準備
... 

## 損失関数の定義（今回は単純な二乗誤差）
... 

## optimizerを準備
... 

## Trainerを準備
... 

## 学習を実行
...


## 6. 可視化付きの訓練

継承とデコレータを使って，`Trainer`を拡張することができる

In [ ]:
import importlib
import jkj1a.training

importlib.reload(jkj1a.training)
from jkj1a.training import animate_step, animate

class AnimatedTrainer(Trainer):
    @animate_step
    def train_step(self, model, data, loss_fn, optimizer):
        return super().train_step(model, data, loss_fn, optimizer)

    @animate
    def train(self, n_epochs=10, print_interval=100):
        return super().train(
            n_epochs=n_epochs,
            print_interval=print_interval,
        )


**注意！：訓練を再実行する際は，モデルを再生成しないと続きからの訓練になってしまう！**

つまり，以下を再実行してから

```python
model = ReLUWeightedSum(n_hiddens=n_hiddens)
optimizer = torch.optim.SGD(model.parameters(),  # モデルのパラメータを登録（更新されていく）
                            lr=learning_rate,    # 学習率
                            momentum=momentum)   # 慣性
```

以下を実行する必要がある．

```python
atrainer = AnimatedTrainer(model, data, loss_fn, optimizer)
animation = atrainer.train(n_epochs=1000,      # 訓練でデータを何周するか

```

In [ ]:
atrainer = AnimatedTrainer(model, data, loss_fn, optimizer)
animation = atrainer.train(n_epochs=300,          # 訓練でデータを何周するか
                           print_interval=50,     # 何周ごとにloss値を表示するか（アニメーションのステップにも関係）
                           with_animation=True,   # アニメーションを表示するか
                           animation_interval=50, # アニメーションのステップ間隔
                           show_sliders=True)     # パラメータのスライダーを表示するか
                           

# 表示された画面の▶を押すと再生します（アニメーションの表示まで少し時間がかかります）。
display(animation)


# 7. 演習

`04-relu-functions.ipynb`で登場した6つの関数に対してそれぞれ学習を行ってみよう
- すでに`target_function`で定義できる
- 学習がなかなかうまくいかないケースもあると思う．AIとも相談しながら，改善法を探してみよう．